# 01 - Download do NCBI, Preparação da Base Longa e Auditoria de Overlap

Este notebook consolida e parametriza os passos históricos de `long/download_from_ncbi.ipynb` e `long/prepare database.ipynb`,
auditando os metadados das 4.360 sequências virais obtidas do NCBI Nucleotide, quantificando o overlap de genomas de origem
contra a base de treinamento do Kraken2 (`viral_cnn.inference.audit_long:quantify_training_overlap`) e documentando o viés de amostragem de leituras (AD-010).

**Invariantes científicas (`AD-004`, `AD-010`):**
- As sequências do NCBI coincidem com as referências do banco de treino (mesmo genoma de origem); o desempenho reportado mede acurácia em genomas já vistos (`seen_genome`) e não generalização para genomas não vistos.
- A função de simulação (`calculate_n_reads`) gerou 500 reads para vírus gigantes (>100 kb) vs. 50 reads para pequenos vírus (<10 kb), introduzindo viés de representação.


In [ ]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import setup_notebook_runtime, read_parquet_sample, read_parquet_hash_sample, reserve_c02_long_run
from viral_cnn.inference.audit_long import quantify_training_overlap

# 1. Inicialização do runtime e contexto de execução (AD-017)
ctx = setup_notebook_runtime()
project_root = next(r for r in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (r / "src" / "viral_cnn").exists())
run_id, output_dir = reserve_c02_long_run(project_root, "long/01")
ctx.run_id, ctx.output_dir = run_id, output_dir

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


## 2. Auditoria de Proveniência e Quantificação de Sobreposição com Treino

Quantifica o compartilhamento de k-mers canônicos ($k=15$) entre as sequências longas e o banco de treinamento,
e documenta o viés de amostragem por faixa de tamanho de genoma.


In [ ]:
# Inventário observado; fixtures sintéticas nunca são promovidas a evidência empírica.
from viral_cnn.inference.audit_long import audit_long_inputs, sample_long_queries, quantify_training_overlap_detailed
long_parquet = ctx.topology.paths["long"]["ncbi_parquet_dir"]
short_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
if ctx.is_smoke_test:
    rng = np.random.RandomState(42)
    bases = ["A", "C", "G", "T"]
    query_df = pd.DataFrame({"query_id": [f"smoke_{i:03d}" for i in range(12)], "sequence": ["".join(rng.choice(bases, size=n)) for n in [400, 600, 999, 2000, 4000, 6000, 10001, 12000, 15000, 20000, 25000, 30000]], "truth_label": [i % 3 for i in range(12)]})
    train_df = pd.DataFrame({"sequence": ["".join(rng.choice(bases, size=512)) for _ in range(20)]})
    query_sampling = {"kind":"synthetic_fixture", "planned_count":12, "realized_count":12, "criterion":"fixed deterministic CPU fixture", "seed":42, "selected_query_ids":query_df.query_id.tolist()}
    train_sampling = {"kind":"synthetic_fixture", "planned_count":20, "realized_count":20, "criterion":"fixed deterministic CPU fixture", "seed":42}
    train_seqs = train_df.sequence.tolist()
else:
    if not Path(long_parquet).exists() or not short_parquet_dir.exists():
        raise FileNotFoundError(f"C02 empirical inputs missing: long={long_parquet}; train={short_parquet_dir}")
    df_long = read_parquet_sample(long_parquet, columns=["sequence", "species_id", "query_id", "read_id"], max_rows=None, split="test")
    available_query_rows=len(df_long)
    missing_sequence=int(df_long.sequence.isna().sum()) if "sequence" in df_long else available_query_rows
    missing_species=int(df_long.species_id.isna().sum()) if "species_id" in df_long else available_query_rows
    query_df = df_long.dropna(subset=["sequence", "species_id"]).copy().reset_index(drop=True)
    query_df["query_id"] = query_df.get("query_id", query_df.get("read_id", query_df.index.astype(str))).astype(str)
    if query_df.query_id.duplicated().any(): raise ValueError("Long query IDs are duplicated; C02 requires stable unique query_id")
    query_df["truth_label"] = query_df["species_id"]
    query_df, query_sampling = sample_long_queries(query_df, limit=int(os.getenv("C02_QUERY_LIMIT", "0")) or None, seed=42)
    query_sampling.update({"available_count":available_query_rows,"excluded_count":available_query_rows-len(df_long.dropna(subset=["sequence","species_id"]))})
    train_limit=int(os.getenv("C02_TRAIN_LIMIT","20000")) or None
    train_df,train_sampling=read_parquet_hash_sample(short_parquet_dir,columns=["sequence_padded","padded_Representation","sequence"],limit=train_limit,seed=42,split="train")
    if train_df.query_id.duplicated().any(): raise ValueError("Training fragment stable IDs are duplicated")
    train_seqs=train_df.sequence.astype(str).tolist()
input_path = str(long_parquet) if not ctx.is_smoke_test else None
input_audit = audit_long_inputs(query_df, query_id_col="query_id", taxon_col="truth_label", source_path=input_path, data_kind="synthetic_fixture" if ctx.is_smoke_test else "NCBI contigs", excluded_count=int(query_sampling.get("excluded_count",0)), exclusion_rule="drop rows missing sequence or species_id", exclusion_counts={"missing_sequence":int(missing_sequence) if not ctx.is_smoke_test else 0,"missing_species_id":int(missing_species) if not ctx.is_smoke_test else 0}, sample_metadata=query_sampling)
input_audit["train_sampling"] = train_sampling
input_audit["query_sampling"] = query_sampling
train_input_audit=audit_long_inputs(train_df,sequence_col="sequence",query_id_col="query_id",taxon_col="truth_label",source_path=str(short_parquet_dir) if not ctx.is_smoke_test else None,data_kind="synthetic_training_fragments" if ctx.is_smoke_test else "training_fragments",excluded_count=int(train_sampling.get("excluded_count",0)),exclusion_rule="drop rows without a canonical sequence",sample_metadata=train_sampling)
overlap_metrics = quantify_training_overlap_detailed(query_df.sequence.astype(str).tolist(), train_seqs, k=15)
overlap_metrics["train_sampling"] = train_sampling
overlap_metrics["query_sampling"] = query_sampling
audit_csv = output_dir / "long_database_preparation_audit.csv"
pd.DataFrame([{"length_bin": name, "observed_query_count": int(((query_df.sequence.str.len() >= lo) & (query_df.sequence.str.len() < hi)).sum())} for name,lo,hi in [("lt_10000",0,10000),("10000_to_99999",10000,100000),("ge_100000",100000,float("inf"))]]).to_csv(audit_csv,index=False)
manifest_01 = {"notebook":"01_ncbi_download_and_prepare_database.ipynb", "long_sequence_protocol":"c02_audit_v1", "is_smoke_test":ctx.is_smoke_test, "input_audit":input_audit, "train_input_audit":train_input_audit, "overlap_metrics":overlap_metrics, "historical_rule":input_audit["historical_sampling_rule"], "artifacts":{"audit_csv":str(audit_csv)}}
manifest_path=output_dir / "manifest_01_ncbi_prep.json"
manifest_path.write_text(json.dumps(manifest_01,indent=2),encoding="utf-8")


## 3. Emissão do Manifesto de Preparação da Base Longa


In [ ]:
print(f"Observed input audit: {input_audit['eligible_count']} eligible, {input_audit['excluded_count']} excluded; genome IDs: {input_audit['genome_identity']['status']}")
print(f"Overlap: {overlap_metrics}")
print(f"Manifest: {manifest_path}")

from viral_cnn.notebook_runtime import write_notebook_manifest
write_notebook_manifest("long/01", ctx.data_root, ctx.is_smoke_test, metrics_summary={"eligible":input_audit["eligible_count"],"overlap":overlap_metrics}, input_files=[input_path] if input_path and Path(input_path).exists() else [], run_id=ctx.run_id, configuration={"protocol":"c02_audit_v1","sampling":query_sampling}, outputs=[audit_csv, manifest_path], output_json_path=output_dir/"run_manifest.json")
